# French emotion classification

**Source:** original master notebook, section S9 (cells 76-78: classical models), *Part A* (cell 80: GRU/LSTM/BiLSTM),
*Part B* (cell 82: CamemBERT), *Part C & D* (cell 84: benchmark) and the duplicate "S4 - French RNN" cell 86.

## 1. Objective and experimental scope
Six-class emotion classification of French text from a **local CSV** (`emotions_fr_final.csv`, 29,286 rows in the original run, not distributed here).

| Block | Protocol in the original notebook |
|---|---|
| Classical (S9) | `prepare_emotion` → 28,996 rows; stratified **80/20**; TF-IDF (50k, 1-2 grams); LogReg + LinearSVM |
| Part A - RNNs | stratified **70/15/15**; training subsampled to 20,000; class-weighted CE; early stopping on validation loss |
| Part B - CamemBERT | 5,000 training rows, 1,000 validation rows, full 15 % test; 2 epochs |
| Part C/D | classical rows + RNN/CamemBERT rows in one table / bar plot |
| S4 legacy | a second RNN protocol (80/20, no validation, no class weights, max length 64) - redundant, no recorded output |

> **Important limitations of the original experiment (kept, not fixed):**
> 1. Classical models use an 80/20 split; Part A/B use a 70/15/15 split → the benchmark in Part C/D compares models on **different test sets**.
> 2. **Only the classical (S9) results were recorded** in the original notebook's outputs. The outputs of Part A, B, C/D and S4 are empty
>    (cells 80, 82, 84, 86), so no RNN or CamemBERT score for French can be reported from the source.
> 3. The English-label set is assumed to be the same six emotions; the CSV's label names are never printed in the source.
>
> **Status:** this notebook has not been executed in the preparation environment (the CSV is private; no GPU/PyTorch there).

In [ ]:
import sys
from pathlib import Path

# Locate the repository root (works from the repo root or from notebooks/) and make `src` importable.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").is_file())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import get_paths, get_device, set_seed, save_table, save_figure

paths = get_paths(create=True)      # data/, results/, cache/, checkpoints/ (override with ITC_* env vars)
SEED = 42
set_seed(SEED)                      # seed once, as the original notebook did
DEVICE = get_device()
print("repo root:", ROOT, "| device:", DEVICE)

## 2. Dataset loading and validation
Put `emotions_fr_final.csv` in the data directory (default `./data/`, or `ITC_DATA_DIR`). Required columns: `emotion` and `fr_text_cleaned`
(or `text`). The third column of the file is never used by the notebook and its meaning is unknown.

In [ ]:
from src.data import load_french_emotions
from src.data.emotion import french_text_column

FR_CSV = paths.data_dir / "emotions_fr_final.csv"
french_emotions = load_french_emotions(FR_CSV)
print(french_emotions.shape, list(french_emotions.columns))

counts = french_emotions["emotion"].value_counts()
dist = pd.DataFrame({"Count": counts, "Percentage (%)": (counts / len(french_emotions) * 100).round(2)})
display(dist)

txt = french_text_column(french_emotions)
print("Missing values:", int(french_emotions.isnull().sum().sum()),
      "| duplicated texts:", int(french_emotions.duplicated(subset=[txt]).sum()),
      "| empty texts:", int((french_emotions[txt].astype(str).str.strip() == "").sum()))
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(data=french_emotions, x="emotion", order=counts.index, hue="emotion", legend=False, palette="magma", ax=ax)
ax.set_title("Distribution des Émotions (Français)"); ax.tick_params(axis="x", rotation=45)
save_figure(fig, paths.figures_dir, "french_emotion", "class_distribution"); plt.show()

## 3. Preprocessing
`prepare_emotion`: keep `fr_text_cleaned` + `emotion`, drop NaN, strip, drop empty texts and duplicate `(text, emotion)` pairs, drop classes with ≤ 50 samples.
The English character filter is **not** applied (it would delete accented letters).

In [ ]:
from sklearn.preprocessing import LabelEncoder
from src.preprocessing import prepare_emotion

emotion_fr = prepare_emotion(french_emotions, "fr_text_cleaned")
le_fr = LabelEncoder()
y_fr = le_fr.fit_transform(emotion_fr["emotion"])
texts_fr = emotion_fr["text"].tolist()
print("rows after preparation:", len(emotion_fr), "| classes:", list(le_fr.classes_))

## 4-6. Classical models (S9) - 80/20 split, test evaluation

In [ ]:
from src.data import split_train_test
from src.features import EMOTION_PIPELINE_TFIDF_CFG, fit_transform_tfidf
from src.models import emotion_pipeline_models
from src.evaluation import compute_emotion_pipeline_metrics, text_classification_report

X_tr, X_te, y_tr, y_te = split_train_test(emotion_fr["text"], y_fr, test_size=0.2, seed=SEED)
vec, Xtr, (Xte,) = fit_transform_tfidf(X_tr, [X_te], **EMOTION_PIPELINE_TFIDF_CFG)

rows = []
for name, clf in emotion_pipeline_models().items():
    clf.fit(Xtr, y_tr); pred = clf.predict(Xte)
    rows.append(compute_emotion_pipeline_metrics(y_te, pred, name, "FR"))
    # NOTE: the original printed a single report titled "LogReg" that was actually computed from the LAST model fitted (LinearSVM).
    # Here every report carries the name of the model that produced it (presentation fix only; no metric changes).
    print(f"\n=== Classification report FR ({name}) ===")
    print(text_classification_report(y_te, pred, le_fr.classes_))
res_fr = pd.DataFrame(rows); display(res_fr)
save_table(res_fr, paths.tables_dir, "french_emotion", "classical_80_20", split="test")

## Part A - French recurrent networks (GRU, LSTM, BiLSTM)
Config `FRENCH_RNN_PART_A`: train subsample 20,000, vocab 20,000, max length 100, batch 64, Adam 1e-3, ≤ 5 epochs,
`ReduceLROnPlateau(patience=1)` and early stopping (patience 2) on validation loss; **class-weighted** cross-entropy
(`compute_class_weight("balanced")`); the best-validation weights are restored before the test evaluation.
The right-padding / last-position caveat described in `docs/methodology.md` §7 applies here too.

In [ ]:
import torch, torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from src.data import split_holdout_test_then_val, subsample_stratified
from src.preprocessing import build_vocab, encode_texts
from src.preprocessing.sequences import embedding_vocab_size
from src.models.rnn import FRENCH_RNN_PART_A as A, RNNClassifier, make_loader, predict_classes, train_with_early_stopping
from src.evaluation import french_benchmark_row, plot_confusion_matrix

X_train, X_val, X_test, y_train, y_val, y_test = split_holdout_test_then_val(texts_fr, y_fr, 0.15, 0.15, seed=SEED)
if len(X_train) > A["train_size"]:
    X_train, y_train = subsample_stratified(X_train, y_train, A["train_size"], seed=SEED)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

vocab = build_vocab(X_train, A["vocab_size"])
loaders = {k: make_loader(encode_texts(X, vocab, A["max_len"]), y, A["batch_size"], shuffle=(k == "train"))
           for k, (X, y) in dict(train=(X_train, y_train), val=(X_val, y_val), test=(X_test, y_test)).items()}
weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
weights_t = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

results_dl = []
for rnn_type in A["rnn_types"]:
    print(f"\nTraining {rnn_type}...")
    model = RNNClassifier(rnn_type, embedding_vocab_size(vocab), A["embed_dim"], A["hidden_dim"], num_outputs=6, dropout=A["dropout"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=A["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=A["scheduler_patience"])
    info = train_with_early_stopping(model, loaders["train"], loaders["val"], nn.CrossEntropyLoss(weight=weights_t),
                                     optimizer, scheduler, A["epochs"], A["early_stopping_patience"], DEVICE,
                                     checkpoint_path=paths.checkpoints_dir / f"french_{rnn_type.lower()}_best.pt")
    preds, t_inf = predict_classes(model, loaders["test"], DEVICE)
    row = french_benchmark_row(y_test, preds, rnn_type, info["train_time"], t_inf)
    results_dl.append(row)
    print({k: round(v, 4) for k, v in row.items() if k != "Model"}, "| stopped early:", info["stopped_early"])
    fig = plot_confusion_matrix(y_test, preds, le_fr.classes_, f"Confusion Matrix - {rnn_type}", figsize=(6, 5), xlabel="", ylabel="")
    save_figure(fig, paths.figures_dir, "french_emotion", "confusion_matrix", model=rnn_type, split="test"); plt.close(fig)

df_dl_res = pd.DataFrame(results_dl); display(df_dl_res)
save_table(df_dl_res, paths.tables_dir, "french_emotion", "dl_comparison", split="test")

## Part B - CamemBERT fine-tuning
Config `FRENCH_CAMEMBERT`: 5,000 training rows drawn from the (already subsampled) training set, first 1,000 validation rows, **full** 15 % test set,
max length 128, batch 4, 2 epochs, lr 2e-5, checkpoint selected on validation macro-F1.

*Deviation (reproducibility only):* the original drew the 5,000 rows with an un-reseeded `np.random.choice`, so the exact subset of the original run is
unrecoverable. Here `np.random.seed(SEED)` is called right before the draw. The original also wrapped the cell in `try/except` and printed
"FAILED"; here exceptions propagate.

In [ ]:
from src.models.hf_finetuning import FRENCH_CAMEMBERT as B, fine_tune_sequence_classifier, macro_f1_metric

np.random.seed(SEED)
idx = np.random.choice(len(X_train), min(B["train_size"], len(X_train)), replace=False)
out = fine_tune_sequence_classifier(
    model_name=B["model_name"],
    train_texts=[X_train[i] for i in idx], train_labels=[int(y_train[i]) for i in idx],
    val_texts=X_val[:B["val_size"]], val_labels=[int(v) for v in y_val[:B["val_size"]]],
    test_texts=X_test, test_labels=[int(v) for v in y_test],
    num_labels=6, training_kwargs=B["training_kwargs"], output_dir=str(paths.checkpoints_dir / "camembert"),
    max_length=B["max_length"], compute_metrics=macro_f1_metric, label_column="label",
    use_padding_collator=False, cpu=B["cpu"],
)
res_trf = french_benchmark_row(out["y_true"], out["y_pred"], "CamemBERT", out["train_time"], out["inference_time"])
df_trf_res = pd.DataFrame([res_trf]); display(df_trf_res)
save_table(df_trf_res, paths.tables_dir, "french_emotion", "camembert", split="test")
fig = plot_confusion_matrix(out["y_true"], out["y_pred"], le_fr.classes_, "Confusion Matrix - CamemBERT", cmap="Greens", figsize=(6, 5), xlabel="", ylabel="")
save_figure(fig, paths.figures_dir, "french_emotion", "confusion_matrix", model="camembert", split="test"); plt.close(fig)

## Part C & D - benchmark table and plot
**Caution:** classical rows come from the 80/20 split, RNN/CamemBERT rows from the 70/15/15 split → different test samples and sizes
(about 5,800 vs about 4,350 rows). Read the plot as indicative only.

In [ ]:
from src.evaluation import combine_french_benchmark

df_dl_trf = pd.concat([df_dl_res, df_trf_res], ignore_index=True)
bench = combine_french_benchmark(res_fr, df_dl_trf)
display(bench)
save_table(bench, paths.tables_dir, "french_emotion", "complete_benchmark", split="test")
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=bench, x="Model", y="Macro-F1", hue="Model", legend=False, palette="viridis", ax=ax)
ax.set_title("French Emotion Complete Benchmark - Macro F1"); plt.xticks(rotation=45); plt.tight_layout()
save_figure(fig, paths.figures_dir, "french_emotion", "complete_benchmark_macro_f1", split="test"); plt.show()

## Optional - legacy "S4" French RNN protocol (original cell 86)
Duplicate of Part A with a different protocol: 80/20 split, **no validation set, no class weights, no early stopping**, max length 64, vocab 10,000, 5 epochs,
test evaluation only. It has no recorded output and is **not** used by the benchmark. Disabled by default.

In [ ]:
RUN_LEGACY_S4 = False
if RUN_LEGACY_S4:
    from src.models.rnn import FRENCH_RNN_S4_LEGACY as L, train_fixed_epochs
    Xtr_l, Xte_l, ytr_l, yte_l = split_train_test(texts_fr, y_fr, test_size=0.2, seed=SEED)
    voc = build_vocab(Xtr_l, L["vocab_size"])
    tr = make_loader(encode_texts(Xtr_l, voc, L["max_len"]), ytr_l, L["batch_size"], shuffle=True)
    te = make_loader(encode_texts(Xte_l, voc, L["max_len"]), yte_l, L["eval_batch_size"])
    legacy = []
    for rnn_type in L["rnn_types"]:
        m = RNNClassifier(rnn_type, embedding_vocab_size(voc), L["embed_dim"], L["hidden_dim"], num_outputs=len(le_fr.classes_), dropout=L["dropout"]).to(DEVICE)
        _, t_tr = train_fixed_epochs(m, tr, nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), lr=L["lr"]), L["epochs"], DEVICE)
        p, t_inf = predict_classes(m, te, DEVICE)
        legacy.append(compute_emotion_pipeline_metrics(yte_l, p, rnn_type, "FR"))
    display(pd.DataFrame(legacy))

## 8. Interpretation (only what was recorded)
Recorded in the original outputs for the classical block (`results/tables/french_emotion__classical_80_20__test__src-cell78.csv`):

| Model | Accuracy | Macro-F1 | Weighted-F1 |
|---|---|---|---|
| LogReg | 0.6795 | 0.6560 | 0.6830 |
| LinearSVM | 0.7202 | 0.6894 | 0.7193 |

* The Linear SVM is ahead of the logistic regression on all three metrics in this single run.
* In the LinearSVM report (the one printed in the original under the title "LogReg"), `surprise` has the lowest F1 (0.50) and `joy`/`sadness` the highest (0.79 / 0.80).
* French scores are well below the English S9 scores (macro-F1 0.656-0.689 vs 0.845-0.872) but the two corpora differ in size, label balance and (per the notebook) possible translation artefacts, so the gap is not attributable to language alone.
* **No result for the GRU/LSTM/BiLSTM or CamemBERT French experiments exists in the source notebook**; any such number must come from running the cells above.